# Preprocessing exploration

Visual walk-through of `bayes_cep.preprocessing`: converts the single retained patient's raw fiber data (`example_data/raw/mesh.vtu`, `example_data/raw/fiber_field.npy`, `example_data/raw/basis_vecs.npy`) into a ground-truth fiber-angle field, a constant prior mean, and synthetic noisy activation-time observations.

This notebook is for interactive/visual inspection only. `pixi run python scripts/preprocess_data.py --data-dir example_data` runs the same pipeline non-interactively and caches its outputs to `example_data/preprocessing/` (see that script's docstring, or `--help`, for the full set of tunable parameters).

In [1]:
from pathlib import Path

import numpy as np
import pyvista as pv

from bayes_cep.mesh.io import load_pyvista_mesh
from bayes_cep.posterior.eikonal_map import EikonalParameterToSolutionMap, EikonalSolverSettings
from bayes_cep.preprocessing.angle_field import build_angle_field_from_fiber_field
from bayes_cep.preprocessing.prior_mean import build_constant_prior_mean
from bayes_cep.preprocessing.synthetic_observations import (
    ObservationSamplingSettings,
    generate_synthetic_observations,
)

pv.set_jupyter_backend("trame")

DATA_DIR = Path.cwd().parent / "example_data"
RAW_DIR = DATA_DIR / "raw"

## Raw data

In [2]:
mesh = load_pyvista_mesh(RAW_DIR / "mesh.vtu")
fiber_field = np.load(RAW_DIR / "fiber_field.npy")
basis_vectors = np.load(RAW_DIR / "basis_vecs.npy")

print(f"{mesh.n_points} vertices, {mesh.n_cells} triangles")
print(f"fiber_field {fiber_field.shape}, basis_vectors {basis_vectors.shape}")

15820 vertices, 31375 triangles
fiber_field (31375, 3), basis_vectors (31375, 3, 2)


In [3]:
cell_centers = mesh.cell_centers()
cell_centers.point_data["fiber_field"] = fiber_field
glyphs = cell_centers.glyph(orient="fiber_field", scale=False, factor=0.5, geom=pv.Arrow())

plotter = pv.Plotter()
plotter.add_mesh(glyphs, color="blue")
plotter.add_mesh(mesh, color="lightgray")
plotter.show()

Widget(value='<iframe src="http://localhost:42647/index.html?ui=P_0x7f777016f770_0&reconnect=auto" class="pyvi…

## Ground-truth fiber-angle field

Per-simplex fiber vectors converted to angles, area-weight-interpolated to vertices, then
re-branched onto a single $\pi$-periodic branch (see `build_angle_field_from_fiber_field`'s
docstring for the exact steps and their caveats).

In [4]:
ground_truth_angle_field = build_angle_field_from_fiber_field(mesh, fiber_field, basis_vectors)

plot_mesh = mesh.copy()
plot_mesh.point_data["angle"] = ground_truth_angle_field
plotter = pv.Plotter()
plotter.add_mesh(plot_mesh, scalars="angle", cmap="hsv", interpolate_before_map=False)
plotter.show()

Widget(value='<iframe src="http://localhost:42647/index.html?ui=P_0x7f770be8e490_1&reconnect=auto" class="pyvi…

## Prior mean

With only one patient, the cross-patient ensemble mean the legacy pipeline used is unavailable; the
prior mean here is the spatial mean of the ground truth, broadcast as a constant field (see
`build_constant_prior_mean`'s docstring).

In [5]:
prior_mean_angle_field = build_constant_prior_mean(ground_truth_angle_field)
print(f"constant prior mean angle: {prior_mean_angle_field[0]:.4f} rad")

constant prior mean angle: 1.0405 rad


## Synthetic ground truth (from the SPDE prior)

An alternative to the real-data ground truth above: draw a ground-truth field directly from the
same Bilaplacian SPDE prior used for inference, fully decoupled from the one retained patient's
data. Uses the legacy reference settings ($\kappa=5$, $\tau=0.01$, seed 0).

In [6]:
from bayes_cep.preprocessing.ground_truth import (
    SyntheticGroundTruthSettings,
    build_synthetic_ground_truth,
)

synthetic_ground_truth = build_synthetic_ground_truth(
    mesh, SyntheticGroundTruthSettings(kappa=0.1, tau=10, seed=0)
)
synthetic_prior_mean = build_constant_prior_mean(synthetic_ground_truth)
print(f"synthetic prior mean angle: {synthetic_prior_mean[0]:.4f} rad")

plot_mesh = mesh.copy()
plot_mesh.point_data["angle"] = synthetic_ground_truth
plotter = pv.Plotter()
plotter.add_mesh(plot_mesh, scalars="angle", cmap="hsv", interpolate_before_map=False)
plotter.show()

synthetic prior mean angle: -0.0509 rad


Widget(value='<iframe src="http://localhost:42647/index.html?ui=P_0x7f76ec1f5450_2&reconnect=auto" class="pyvi…

## Synthetic activation-time observations

Forward-solve the eikonal equation on the ground-truth fiber field, then sample a sparse, noisy
subset of vertices as synthetic data for the likelihood.

In [7]:
forward_map = EikonalParameterToSolutionMap(
    mesh,
    basis_vectors,
    EikonalSolverSettings(
        initial_site_ind=12650,
        longitudinal_velocity=1.5,
        transversal_velocity=1.0,
    ),
)
activation_times = forward_map.evaluate_forward(ground_truth_angle_field)

observed_vertex_indices, observed_activation_times = generate_synthetic_observations(
    forward_map,
    ground_truth_angle_field,
    ObservationSamplingSettings(num_observations=1000, noise_variance=1e-3, seed=0),
)
print(f"{observed_vertex_indices.shape[0]} observed vertices")

1000 observed vertices


In [8]:
plot_mesh = mesh.copy()
plot_mesh.point_data["activation_time"] = activation_times
observation_points = pv.PolyData(mesh.points[observed_vertex_indices])

plotter = pv.Plotter()
plotter.add_mesh(plot_mesh, scalars="activation_time", cmap="viridis")
plotter.add_mesh(observation_points, color="red", point_size=8, render_points_as_spheres=True)
plotter.show()

Widget(value='<iframe src="http://localhost:42647/index.html?ui=P_0x7f76b3ef0550_3&reconnect=auto" class="pyvi…